# Final Manning's N and Infiltration Analysis

This notebook demonstrates how to extract and analyze the **final resolved** Manning's N and infiltration parameters from HEC-RAS geometry HDF files.

## What You'll Learn
- Read preprocessed per-cell Manning's N and infiltration values
- Analyze the Manning's N calibration table (base + region overrides)
- Compare base vs calibrated Manning's N values
- Compute statistics on final parameter layers

In [ ]:
# Setup
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ras_commander.hdf.HdfLandCover import HdfLandCover
from ras_commander.hdf.HdfInfiltration import HdfInfiltration
print("Imports successful")

In [ ]:
# Set geometry HDF path
geom_hdf = Path(r"G:\BayouConway_A14_03202026\BayouConway_Update_250731.g17.hdf")
assert geom_hdf.exists()
print(f"Geometry: {geom_hdf.name}")

## Preprocessed Per-Cell Manning's N
The exact per-cell values HEC-RAS uses in computation.

In [ ]:
mann_df = HdfLandCover.get_preprocessed_mannings_n(geom_hdf)
print(f"Cells: {len(mann_df):,}")
print(f"N range: {mann_df["mannings_n"].min():.4f} to {mann_df["mannings_n"].max():.4f}")
print(f"Mean: {mann_df["mannings_n"].mean():.4f}")
stats = HdfLandCover.get_preprocessed_mannings_stats(geom_hdf)
stats

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(mann_df["mannings_n"], bins=50, edgecolor="black", alpha=0.7, color="steelblue")
ax.set_xlabel("Manning's N")
ax.set_ylabel("Cell Count")
ax.set_title("Distribution of Manning's N Values")
ax.axvline(mann_df["mannings_n"].mean(), color="red", linestyle="--", label=f"Mean={mann_df["mannings_n"].mean():.4f}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Calibration Table and Base vs Calibrated Comparison

In [ ]:
cal = HdfLandCover.get_mannings_calibration_table(geom_hdf)
print(f"Land cover classes: {len(cal)}, Calibration regions: {len(cal.columns) - 2}")
print()
print("Base values:")
print(cal.iloc[:, :2].to_string(index=False))

In [ ]:
comp = HdfLandCover.compare_base_vs_calibrated(geom_hdf)
if comp is not None and len(comp) > 0:
    print(f"Override entries: {len(comp)}, Classes: {comp["land_cover_class"].nunique()}, Regions: {comp["region_name"].nunique()}")
    print()
    print("Largest N increases:")
    print(comp[comp["delta_n"] > 0].nlargest(5, "delta_n")[["land_cover_class", "region_name", "base_n", "calibrated_n", "delta_n"]].to_string(index=False))

## Preprocessed Infiltration Parameters

In [ ]:
for var in ["Curve Number", "Abstraction Ratio", "Minimum Infiltration Rate"]:
    df = HdfInfiltration.get_preprocessed_infiltration(geom_hdf, variable=var)
    if len(df) > 0:
        print(f"{var}: {len(df):,} cells, range {df["value"].min():.3f}-{df["value"].max():.3f}, mean {df["value"].mean():.3f}")
    else:
        print(f"{var}: No data")

In [ ]:
cn = HdfInfiltration.get_preprocessed_infiltration(geom_hdf, variable="Curve Number")
if len(cn) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(cn["value"], bins=50, edgecolor="black", alpha=0.7, color="forestgreen")
    ax.set_xlabel("SCS Curve Number")
    ax.set_ylabel("Cell Count")
    ax.set_title("Distribution of Curve Number Values")
    ax.axvline(cn["value"].mean(), color="red", linestyle="--", label=f"Mean={cn["value"].mean():.1f}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Key Takeaways

-  reads the exact per-cell N values from geometry HDF
-  reads per-cell infiltration parameters
-  shows the full base + override matrix
-  highlights calibration changes
- For raster export:  produces a GeoTIFF